# 2026-10-11 — XFeat 이후 5가지 매칭 방법 통합 비교

1. 공식 XFeat float32 cosine MNN (`min_cossim=-1`).
2. 기존 성공: 고정 분위수 8-bin → 직접 thermometer → AVX2 SIMD4 + fused 양방향 MNN.
3. 같은 분위수 8-bin → 000~111 정수 → ABS/Manhattan 거리. **Manhattan Hashing 논문 전체 파이프라인이 아닙니다.**
4. 학습한 8개 float 비트 → 차원별 256→8 LUT → 정수 ABS.
5. 같은 학습 비트/LUT → thermometer → 기존 AVX2 SIMD4 MNN.

모든 방법은 동일한 XFeat descriptor와 held-out test 이미지 쌍을 사용합니다. XFeat는 동결합니다.
분위수/LUT는 학습 이미지에서만 만들고 모든 평가점에 재사용합니다. ABS와 thermometer의 MNN 일치를 자동 검사합니다.
CPU runtime (AVX2 x86), torch와 C++ 모두 2 threads. **속도는 XFeat descriptor 이후**이며, 인코딩·LUT·packing·B의 SIMD 배열 전환·MNN을 포함합니다.

용량은 descriptor당 저장 바이트, 매칭 작업 바이트, 공유 모델 바이트를 구분합니다.
정수 3-bit 저장은 24 B지만 매칭 배열은 64 B입니다. `packed_storage_total_ms`는 packing/unpacking 포함 시간입니다.
Thermometer는 56 B, float는 256 B입니다. 새 LUT 공유 비용은 16,384 B + 비트 위치 32 B입니다.
학습시간과 1/1,000/10,000쌍 처리 총시간도 표시합니다. table 학습비는 두 새 방법이 공동으로 한 번 부담합니다.

## 실행 방법
Colab에서 **런타임 유형 CPU → 모두 실행**. 이전 실험의 descriptor 캐시가 있으면 그대로 재사용하고 비트/table을 다시 학습합니다.
캐시가 없으면 기존 `CSE527_datasets/val2017.zip`으로 COCO 24장, 원본 이미지별 16/3/5 train/validation/test 분리, 각 3개 변환을 생성합니다.
기본 실행에 HPatches 전체 116개 시퀀스 × 5 = 580쌍 외부 평가를 포함합니다. 합성 15쌍은 빠른 확인용입니다.
결과 표/CSV와 codec은 오늘 날짜 새 폴더에 저장됩니다. 기존 실험 파일은 수정하지 않습니다.


## HPatches 실제 전체 외부 평가 결과

총 **116개 장면, 580쌍**, 조명 57개 + 시점 59개 장면입니다. 학습 모델은 COCO에서 고정했습니다.
정확도는 580쌍 전체, 시간은 조명 12쌍 + 시점 12쌍의 고정 표본을 반복 측정한 중앙값입니다.

| 방법 | 인코딩+MNN ms | 저장 B/KP | Precision 전체 | 조명 | 시점 |
|---|---:|---:|---:|---:|---:|
| float_cosine | 4.091 | 256 | 60.59% | 68.38% | 50.18% |
| quantile_thermometer_AVX2 | 1.016 | 56 | 58.52% | 67.14% | 47.13% |
| quantile_integer_ABS_8bin | 1.705 | 24 | 58.52% | 67.14% | 47.13% |
| learned_bits_integer_ABS | 1.689 | 24 | 57.92% | 66.69% | 46.30% |
| learned_bits_thermometer_AVX2 | 1.787 | 56 | 57.92% | 66.69% | 46.30% |

ABS/thermometer 매칭 일치 검사: 모든 580쌍 통과. 정수 ABS의 packing/unpacking 포함 시간은 기존 1.977ms, 새 방법 1.914ms입니다. 24B 저장 기본 표의 시간은 64B 작업 배열 매칭입니다.

전체 테스트 이미지 수는 696장(116×6)입니다. 최대 변 640 리사이즈 후 <3px precision을 사용한 자체 비교이며 공식 HPatches MMA와 같지 않습니다. 전체 아카이브를 사용했고 일부 장면을 제외하는 프로토콜을 적용하지 않았습니다.

```json
{
  "manifest": {
    "scenes": [
      "i_ajuntament",
      "i_autannes",
      "i_bologna",
      "i_books",
      "i_boutique",
      "i_bridger",
      "i_brooklyn",
      "i_castle",
      "i_chestnuts",
      "i_contruction",
      "i_crownday",
      "i_crownnight",
      "i_dc",
      "i_dome",
      "i_duda",
      "i_fenis",
      "i_fog",
      "i_fruits",
      "i_gonnenberg",
      "i_greenhouse",
      "i_greentea",
      "i_indiana",
      "i_kions",
      "i_ktirio",
      "i_kurhaus",
      "i_leuven",
      "i_lionday",
      "i_lionnight",
      "i_londonbridge",
      "i_melon",
      "i_miniature",
      "i_nescafe",
      "i_nijmegen",
      "i_nuts",
      "i_objects",
      "i_parking",
      "i_partyfood",
      "i_pencils",
      "i_pinard",
      "i_pool",
      "i_porta",
      "i_resort",
      "i_salon",
      "i_santuario",
      "i_school",
      "i_ski",
      "i_smurf",
      "i_steps",
      "i_table",
      "i_tools",
      "i_toy",
      "i_troulos",
      "i_veggies",
      "i_village",
      "i_whitebuilding",
      "i_yellowtent",
      "i_zion",
      "v_abstract",
      "v_adam",
      "v_apprentices",
      "v_artisans",
      "v_astronautis",
      "v_azzola",
      "v_bark",
      "v_bees",
      "v_beyus",
      "v_bip",
      "v_bird",
      "v_birdwoman",
      "v_blueprint",
      "v_boat",
      "v_bricks",
      "v_busstop",
      "v_calder",
      "v_cartooncity",
      "v_charing",
      "v_churchill",
      "v_circus",
      "v_coffeehouse",
      "v_colors",
      "v_courses",
      "v_dirtywall",
      "v_dogman",
      "v_eastsouth",
      "v_feast",
      "v_fest",
      "v_gardens",
      "v_grace",
      "v_graffiti",
      "v_home",
      "v_laptop",
      "v_london",
      "v_machines",
      "v_man",
      "v_maskedman",
      "v_pomegranate",
      "v_posters",
      "v_samples",
      "v_soldiers",
      "v_strand",
      "v_sunseason",
      "v_tabletop",
      "v_talent",
      "v_tempera",
      "v_there",
      "v_underground",
      "v_vitro",
      "v_wall",
      "v_wapping",
      "v_war",
      "v_weapons",
      "v_woman",
      "v_wormhole",
      "v_wounded",
      "v_yard",
      "v_yuri"
    ],
    "n_scenes": 116,
    "n_pairs": 580,
    "illumination_scenes": 57,
    "viewpoint_scenes": 59,
    "max_side": 640,
    "top_k": 1000,
    "resize_homography": "S2 @ H @ inv(S1)",
    "source": "https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip",
    "extraction_seconds": 55.73693216099991,
    "protocol": "all archive sequences, 1 versus 2..6; no excluded sequences; precision <3px, not official MMA benchmark",
    "no_hpatches_training": true
  },
  "environment": {
    "date": "2026-10-11",
    "cpu": "x86_64",
    "platform": "Linux-6.18.44-x86_64-with-glibc2.39",
    "torch_threads": 2,
    "cpp_threads": 2,
    "selected_bits": [
      31,
      25,
      24,
      23,
      22,
      26,
      21,
      20
    ],
    "scope": "CPU after XFeat descriptor; same cached test pairs; transpose included in matching; scalar ABS uses SSE2/OpenMP, thermometer AVX2 SIMD4",
    "precision": "pooled correct/mutual, homography error <3px"
  },
  "summary": [
    {
      "method": "float_cosine",
      "precision_pct": "60.588894870572744",
      "mutual": "291631",
      "correct": "176696",
      "encode_ms": "0.0",
      "matching_ms": "4.078853000237359",
      "total_ms": "4.09137700012252",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "256",
      "working_bytes_per_kp": "256",
      "shared_model_bytes": "0",
      "offline_fit_seconds": "0.0",
      "speedup_vs_float": "1.0",
      "total_for_1_pairs_seconds": "0.00409137700012252",
      "total_for_1000_pairs_seconds": "4.09137700012252",
      "total_for_10000_pairs_seconds": "40.9137700012252"
    },
    {
      "method": "quantile_thermometer_AVX2",
      "precision_pct": "58.524160598458494",
      "mutual": "286603",
      "correct": "167732",
      "encode_ms": "0.06153749995974067",
      "matching_ms": "0.9112185002777551",
      "total_ms": "1.015916500136882",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "56",
      "working_bytes_per_kp": "56",
      "shared_model_bytes": "1792",
      "offline_fit_seconds": "0.2102956970002196",
      "speedup_vs_float": "4.027276847626019",
      "total_for_1_pairs_seconds": "0.2113116135003565",
      "total_for_1000_pairs_seconds": "1.2262121971371016",
      "total_for_10000_pairs_seconds": "10.36946069836904"
    },
    {
      "method": "quantile_integer_ABS_8bin",
      "precision_pct": "58.524160598458494",
      "mutual": "286603",
      "correct": "167732",
      "encode_ms": "0.3532920002271567",
      "matching_ms": "1.3477970003350492",
      "total_ms": "1.7048950001026242",
      "packed_storage_total_ms": "1.9773955000346177",
      "stored_bytes_per_kp": "24",
      "working_bytes_per_kp": "64",
      "shared_model_bytes": "1792",
      "offline_fit_seconds": "0.2102956970002196",
      "speedup_vs_float": "2.399782391218371",
      "total_for_1_pairs_seconds": "0.21200059200032223",
      "total_for_1000_pairs_seconds": "1.9151906971028438",
      "total_for_10000_pairs_seconds": "17.25924569802646"
    },
    {
      "method": "learned_bits_integer_ABS",
      "precision_pct": "57.916258286171974",
      "mutual": "284661",
      "correct": "164865",
      "encode_ms": "0.29837500005669426",
      "matching_ms": "1.292408000153955",
      "total_ms": "1.6890209999473882",
      "packed_storage_total_ms": "1.913568999952986",
      "stored_bytes_per_kp": "24",
      "working_bytes_per_kp": "64",
      "shared_model_bytes": "16416",
      "offline_fit_seconds": "7.919621357999858",
      "speedup_vs_float": "2.4223363713357995",
      "total_for_1_pairs_seconds": "7.921310378999805",
      "total_for_1000_pairs_seconds": "9.608642357947247",
      "total_for_10000_pairs_seconds": "24.80983135747374"
    },
    {
      "method": "learned_bits_thermometer_AVX2",
      "precision_pct": "57.916258286171974",
      "mutual": "284661",
      "correct": "164865",
      "encode_ms": "0.831363500083171",
      "matching_ms": "0.9042179999596556",
      "total_ms": "1.7874745001336123",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "56",
      "working_bytes_per_kp": "56",
      "shared_model_bytes": "16416",
      "offline_fit_seconds": "7.919621357999858",
      "speedup_vs_float": "2.2889148907112764",
      "total_for_1_pairs_seconds": "7.921408832499991",
      "total_for_1000_pairs_seconds": "9.70709585813347",
      "total_for_10000_pairs_seconds": "25.79436635933598"
    }
  ],
  "by_type": [
    {
      "category": "illumination",
      "method": "float_cosine",
      "mutual": "166793",
      "correct": "114048",
      "total_ms": "4.062212999997428",
      "timed_pairs": "12",
      "precision_pct": "68.37697025654553"
    },
    {
      "category": "illumination",
      "method": "quantile_thermometer_AVX2",
      "mutual": "163244",
      "correct": "109594",
      "total_ms": "1.0162920000311715",
      "timed_pairs": "12",
      "precision_pct": "67.1350861287398"
    },
    {
      "category": "illumination",
      "method": "quantile_integer_ABS_8bin",
      "mutual": "163244",
      "correct": "109594",
      "total_ms": "1.6878339999948366",
      "timed_pairs": "12",
      "precision_pct": "67.1350861287398"
    },
    {
      "category": "illumination",
      "method": "learned_bits_integer_ABS",
      "mutual": "162167",
      "correct": "108154",
      "total_ms": "1.693462500043097",
      "timed_pairs": "12",
      "precision_pct": "66.69297699285305"
    },
    {
      "category": "illumination",
      "method": "learned_bits_thermometer_AVX2",
      "mutual": "162167",
      "correct": "108154",
      "total_ms": "1.8091164999987086",
      "timed_pairs": "12",
      "precision_pct": "66.69297699285305"
    },
    {
      "category": "viewpoint",
      "method": "float_cosine",
      "mutual": "124838",
      "correct": "62648",
      "total_ms": "4.225349000080314",
      "timed_pairs": "12",
      "precision_pct": "50.183437735304956"
    },
    {
      "category": "viewpoint",
      "method": "quantile_thermometer_AVX2",
      "mutual": "123359",
      "correct": "58138",
      "total_ms": "1.015916500136882",
      "timed_pairs": "12",
      "precision_pct": "47.129110968798386"
    },
    {
      "category": "viewpoint",
      "method": "quantile_integer_ABS_8bin",
      "mutual": "123359",
      "correct": "58138",
      "total_ms": "1.7334130002382153",
      "timed_pairs": "12",
      "precision_pct": "47.129110968798386"
    },
    {
      "category": "viewpoint",
      "method": "learned_bits_integer_ABS",
      "mutual": "122494",
      "correct": "56711",
      "total_ms": "1.6710989998500736",
      "timed_pairs": "12",
      "precision_pct": "46.29696148382778"
    },
    {
      "category": "viewpoint",
      "method": "learned_bits_thermometer_AVX2",
      "mutual": "122494",
      "correct": "56711",
      "total_ms": "1.771494999729839",
      "timed_pairs": "12",
      "precision_pct": "46.29696148382778"
    }
  ]
}
```


## 실제 통합 실행 결과 — 2026-10-11

동일한 기존 테스트 캐시 15쌍으로 로컬 CPU에서 학습·평가했습니다. Colab 실행 결과는 CPU에 따라 달라집니다.

| 방법 | 인코딩+MNN ms | 저장 B/KP | 작업 B/KP | Precision |
|---|---:|---:|---:|---:|
| float_cosine | 2.392 | 256 | 256 | 71.89% |
| quantile_thermometer_AVX2 | 0.975 | 56 | 56 | 69.15% |
| quantile_integer_ABS_8bin | 1.712 | 24 | 64 | 69.15% |
| learned_bits_integer_ABS | 1.685 | 24 | 64 | 68.40% |
| learned_bits_thermometer_AVX2 | 1.797 | 56 | 56 | 68.40% |

학습 전체 호출(분위수 초기화 포함): 7.920초. 분위수 경계 계산: 0.210초. ABS/thermometer 대응 일치와 SIMD 커널 검증 통과. 새로운 데이터의 XFeat 추출은 이번 캐시 재평가 시간에 포함하지 않았습니다.

정수 방법의 24B 저장을 실제로 packing/unpacking하면 총시간은 기존 ABS 1.980ms, 새 ABS 1.900ms입니다. 위 표의 기본 시간은 64B 작업 배열을 사용하는 경로입니다. 작업 B/KP는 표현 크기이며 최대 임시 메모리 사용량은 아닙니다.

실행 환경 및 상세 측정값은 아래 JSON snapshot에 함께 보존합니다.
```json
{
  "environment": {
    "date": "2026-10-11",
    "cpu": "x86_64",
    "platform": "Linux-6.18.44-x86_64-with-glibc2.39",
    "torch_threads": 2,
    "cpp_threads": 2,
    "selected_bits": [
      31,
      25,
      24,
      23,
      22,
      26,
      21,
      20
    ],
    "scope": "CPU after XFeat descriptor; same cached test pairs; transpose included in matching; scalar ABS uses SSE2/OpenMP, thermometer AVX2 SIMD4",
    "precision": "pooled correct/mutual, homography error <3px"
  },
  "summary": [
    {
      "method": "float_cosine",
      "precision_pct": "71.89214427454185",
      "mutual": "8567",
      "correct": "6159",
      "encode_ms": "0.0",
      "matching_ms": "2.328980000129377",
      "total_ms": "2.391744999840739",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "256",
      "working_bytes_per_kp": "256",
      "shared_model_bytes": "0",
      "offline_fit_seconds": "0.0",
      "speedup_vs_float": "1.0",
      "total_for_1_pairs_seconds": "0.002391744999840739",
      "total_for_1000_pairs_seconds": "2.391744999840739",
      "total_for_10000_pairs_seconds": "23.917449998407392"
    },
    {
      "method": "quantile_thermometer_AVX2",
      "precision_pct": "69.1504854368932",
      "mutual": "8240",
      "correct": "5698",
      "encode_ms": "0.06263400018724496",
      "matching_ms": "0.9257150004486903",
      "total_ms": "0.9750199997142772",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "56",
      "working_bytes_per_kp": "56",
      "shared_model_bytes": "1792",
      "offline_fit_seconds": "0.21029569700021966",
      "speedup_vs_float": "2.453021477038033",
      "total_for_1_pairs_seconds": "0.21127071699993394",
      "total_for_1000_pairs_seconds": "1.1853156967144969",
      "total_for_10000_pairs_seconds": "9.960495694142992"
    },
    {
      "method": "quantile_integer_ABS_8bin",
      "precision_pct": "69.1504854368932",
      "mutual": "8240",
      "correct": "5698",
      "encode_ms": "0.353262000317045",
      "matching_ms": "1.2998790002711758",
      "total_ms": "1.7118000000664324",
      "packed_storage_total_ms": "1.9801339999503398",
      "stored_bytes_per_kp": "24",
      "working_bytes_per_kp": "64",
      "shared_model_bytes": "1792",
      "offline_fit_seconds": "0.21029569700021966",
      "speedup_vs_float": "1.3972105384670634",
      "total_for_1_pairs_seconds": "0.2120074970002861",
      "total_for_1000_pairs_seconds": "1.922095697066652",
      "total_for_10000_pairs_seconds": "17.328295697664544"
    },
    {
      "method": "learned_bits_integer_ABS",
      "precision_pct": "68.40107737512243",
      "mutual": "8168",
      "correct": "5587",
      "encode_ms": "0.3024659999937285",
      "matching_ms": "1.3058980002824683",
      "total_ms": "1.6845179998199455",
      "packed_storage_total_ms": "1.8997530000888219",
      "stored_bytes_per_kp": "24",
      "working_bytes_per_kp": "64",
      "shared_model_bytes": "16416",
      "offline_fit_seconds": "7.9196213579998584",
      "speedup_vs_float": "1.4198393843796193",
      "total_for_1_pairs_seconds": "7.921305875999678",
      "total_for_1000_pairs_seconds": "9.604139357819804",
      "total_for_10000_pairs_seconds": "24.764801356199314"
    },
    {
      "method": "learned_bits_thermometer_AVX2",
      "precision_pct": "68.40107737512243",
      "mutual": "8168",
      "correct": "5587",
      "encode_ms": "0.8334569997714425",
      "matching_ms": "0.9004670000649639",
      "total_ms": "1.7972990003727318",
      "packed_storage_total_ms": "",
      "stored_bytes_per_kp": "56",
      "working_bytes_per_kp": "56",
      "shared_model_bytes": "16416",
      "offline_fit_seconds": "7.9196213579998584",
      "speedup_vs_float": "1.3307440772763621",
      "total_for_1_pairs_seconds": "7.921418657000231",
      "total_for_1000_pairs_seconds": "9.71692035837259",
      "total_for_10000_pairs_seconds": "25.892611361727177"
    }
  ]
}
```


In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
import importlib.util, subprocess, sys
for module, package in {'cv2':'opencv-python-headless','torch':'torch','pandas':'pandas','tqdm':'tqdm','scipy':'scipy','matplotlib':'matplotlib'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
from google.colab import drive
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
drive.mount('/content/drive')
RUN_ID=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S')
BASE=Path('/content/drive/MyDrive/Colab Notebooks')
ROOT=BASE/'2026-10-11_XFeat_5methods_unified_comparison'/RUN_ID
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_CACHE=BASE/'2026-10-11_learned_float_bits_8bin'/'run'
COCO_ZIP=BASE/'CSE527_datasets'/'val2017.zip'
REUSE_PREVIOUS_DESCRIPTORS=True
print('저장 위치:',ROOT)


In [ ]:
sources={'experiment.py': '"""Learn fixed float-bit positions and ordered 8-bin LUTs; compare exact L1 implementations.\nTraining is discrete greedy selection + supervised coordinate descent, not neural training.\n"""\nfrom pathlib import Path\nimport argparse, ctypes, hashlib, json, platform, subprocess, sys, time, urllib.request, zipfile\nimport numpy as np\n\nSEED=20261011\nXFEAT_COMMIT=\'e92685f57f8318b18725c5c8c0bd28c7fe188d9a\'\n\ndef patterns(x, positions):\n    u=np.ascontiguousarray(x,dtype=np.float32).view(np.uint32)\n    c=np.zeros(u.shape,np.uint16)\n    for r,p in enumerate(positions): c |= (((u >> int(p)) & 1).astype(np.uint16) << r)\n    return c\n\ndef bins_from_edges(x, edges):\n    return (x[:,:,None] >= edges[None,:,:]).sum(-1).astype(np.uint8)\n\ndef initial_lut(x, positions, targets):\n    c=patterns(x,positions); size=1<<len(positions)\n    lut=np.zeros((64,size),np.uint8); counts=np.zeros((64,size),np.int32)\n    for d in range(64):\n        n=np.bincount(c[:,d],minlength=size)\n        s=np.bincount(c[:,d],weights=targets[:,d],minlength=size)\n        # Unseen patterns deterministically map to the training dimension\'s mean bin.\n        means=np.full(size,float(targets[:,d].mean()))\n        np.divide(s,n,out=means,where=n>0)\n        lut[d]=np.clip(np.rint(means),0,7).astype(np.uint8); counts[d]=n\n    return lut,counts\n\ndef encode_np(x,positions,lut): return lut[np.arange(64)[None,:],patterns(x,positions)]\n\ndef triplet_metrics(a,p,n):\n    dp=np.abs(a.astype(np.int16)-p.astype(np.int16)).sum(1)\n    dn=np.abs(a.astype(np.int16)-n.astype(np.int16)).sum(1)\n    return dp,dn,float((dp<dn).mean()),float(np.maximum(0,8+dp-dn).mean()/64)\n\ndef train_codec(triplets, calibration, out, nbits=8, search_samples=2000, sweeps=2):\n    rng=np.random.default_rng(SEED); order=rng.permutation(len(triplets[0]))\n    if len(order)<100: raise ValueError(\'Need at least 100 geometrically labelled triplets.\')\n    cut=max(1,int(len(order)*.2)); inner=order[:cut]; fit=order[cut:]\n    fit=fit[:search_samples]; inner=inner[:search_samples]\n    a,p,n=[np.ascontiguousarray(x[fit],np.float32) for x in triplets]\n    va,vp,vn=[np.ascontiguousarray(x[inner],np.float32) for x in triplets]\n    # All quantile teachers, bit selection and LUT updates use training scenes only.\n    edges=np.quantile(calibration,np.arange(1,8)/8,axis=0).T.astype(np.float32)\n    x=np.concatenate([a,p,n]); target=bins_from_edges(x,edges)\n    vtarget=bins_from_edges(np.concatenate([va,vp,vn]),edges)\n    selected=[]; history=[]; started=time.perf_counter()\n    for step in range(nbits):\n        candidates=[]\n        for bit in range(32):\n            if bit in selected: continue\n            positions=selected+[bit]; lut,_=initial_lut(x,positions,target)\n            codes=encode_np(np.concatenate([va,vp,vn]),positions,lut)\n            ca,cp,cn=np.split(codes,3)\n            _,_,rank,loss=triplet_metrics(ca,cp,cn)\n            distill=float(np.square(codes.astype(np.float32)-vtarget).mean())\n            score=loss+.03*distill\n            candidates.append((score,bit,rank,loss,distill))\n        score,bit,rank,loss,distill=min(candidates)\n        selected.append(bit)\n        row=dict(step=step+1,bit=int(bit),score=score,inner_rank=rank,margin_loss=loss,distill_mse=distill)\n        history.append(row); print(\'bit selection\',row,flush=True)\n        (out/\'selection_history.json\').write_text(json.dumps(history,indent=2))\n    # Refit LUT on all sampled training triplets, then supervise its ordered labels.\n    take=order[:min(len(order),8000)]\n    a,p,n=[np.ascontiguousarray(v[take],np.float32) for v in triplets]\n    x=np.concatenate([a,p,n]); target=bins_from_edges(x,edges)\n    lut,counts=initial_lut(x,selected,target)\n    ca,cp,cn=[patterns(v,selected) for v in (a,p,n)]\n    ya,yp,yn=[lut[np.arange(64)[None,:],c].astype(np.int16) for c in (ca,cp,cn)]\n    ta,tp,tn=np.split(target,3); dp,dn,_,_=triplet_metrics(ya,yp,yn)\n    refinement=[]; regularizer=.03\n    for sweep in range(sweeps):\n        changes=0\n        for d in range(64):\n            for c in rng.permutation(np.flatnonzero(counts[d])):\n                ids=np.flatnonzero((ca[:,d]==c)|(cp[:,d]==c)|(cn[:,d]==c))\n                old=int(lut[d,c]); aa=ya[ids,d]; pp=yp[ids,d]; nn=yn[ids,d]\n                oldp=np.abs(aa-pp); oldn=np.abs(aa-nn)\n                maska=ca[ids,d]==c; maskp=cp[ids,d]==c; maskn=cn[ids,d]==c\n                best=(float(\'inf\'),old,None,None,None,None,None)\n                for val in range(8):\n                    na=np.where(maska,val,aa); np_=np.where(maskp,val,pp); nneg=np.where(maskn,val,nn)\n                    ndp=dp[ids]-oldp+np.abs(na-np_); ndn=dn[ids]-oldn+np.abs(na-nneg)\n                    # This is the affected part of the global triplet loss + teacher regularization.\n                    reg=((na-ta[ids,d])**2+(np_-tp[ids,d])**2+(nneg-tn[ids,d])**2).sum()\n                    score=float(np.maximum(0,8+ndp-ndn).sum()+regularizer*reg/3)\n                    if score<best[0]-1e-9 or (abs(score-best[0])<1e-9 and val==old):\n                        best=(score,val,ndp,ndn,na,np_,nneg)\n                if best[1]!=old:\n                    changes+=1;lut[d,c]=best[1];dp[ids]=best[2];dn[ids]=best[3]\n                    ya[ids,d]=best[4];yp[ids,d]=best[5];yn[ids,d]=best[6]\n        row=dict(sweep=sweep+1,changed_entries=changes,train_rank=float((dp<dn).mean()),\n                 margin_loss=float(np.maximum(0,8+dp-dn).mean()/64))\n        refinement.append(row);print(\'LUT training\',row,flush=True)\n    result=dict(positions=selected,nbits=nbits,levels=8,selection_history=history,refinement=refinement,\n                seconds=time.perf_counter()-started,training_triplets=len(triplets[0]),lut_training_triplets=len(a),\n                method=\'greedy bit selection with internal training-only holdout; supervised integer LUT coordinate descent\',\n                lut_scope=\'per descriptor dimension, shared across all images and keypoints\',\n                teacher=\'training-scene per-dimension quantile bins; regularization only\',\n                unseen_pattern_policy=\'training dimension mean bin; never fitted on evaluation data\')\n    np.savez(out/\'learned_codec.npz\',positions=np.array(selected,np.int32),lut=lut,counts=counts,edges=edges)\n    (out/\'training.json\').write_text(json.dumps(result,indent=2))\n    return selected,lut,edges,result\n\nclass Engine:\n    def __init__(self,root):\n        root=Path(root); lib=root/\'bit_codec.so\'\n        subprocess.run([\'g++\',\'-O3\',\'-march=native\',\'-std=c++17\',\'-shared\',\'-fPIC\',str(root/\'bit_codec.cpp\'),\'-o\',str(lib)],check=True)\n        self.lib=ctypes.CDLL(str(lib.resolve())); self.flags=[\'-O3\',\'-march=native\'];self.root=root\n        P=ctypes.c_void_p; I=ctypes.c_int\n        for name,args in {\'encode\':[P,I,P,I,P,P],\'pack3\':[P,I,P],\'unpack3\':[P,I,P],\n                          \'packthermo\':[P,I,P],\'absdist\':[P,P,I,I,P],\'thermodist\':[P,P,I,I,P],\n                          \'absmnn\':[P,P,I,I,P,P],\'thermomnn\':[P,P,I,I,P,P]}.items():\n            f=getattr(self.lib,name);f.argtypes=args;f.restype=None\n    @staticmethod\n    def ptr(a):return a.ctypes.data_as(ctypes.c_void_p)\n    def encode(self,x,pos,lut):\n        x=np.ascontiguousarray(x,np.float32);pos=np.ascontiguousarray(pos,np.int32);lut=np.ascontiguousarray(lut,np.uint8)\n        out=np.empty(x.shape,np.uint8);self.lib.encode(self.ptr(x),len(x),self.ptr(pos),len(pos),self.ptr(lut),self.ptr(out));return out\n    def pack3(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),24),np.uint8)\n        self.lib.pack3(self.ptr(x),len(x),self.ptr(out));return out\n    def unpack3(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),64),np.uint8)\n        self.lib.unpack3(self.ptr(x),len(x),self.ptr(out));return out\n    def thermo(self,x):\n        x=np.ascontiguousarray(x,np.uint8);out=np.empty((len(x),7),np.uint64)\n        self.lib.packthermo(self.ptr(x),len(x),self.ptr(out));return out\n    def distance(self,a,b,thermo=False):\n        out=np.empty((len(a),len(b)),np.uint16)\n        getattr(self.lib,\'thermodist\' if thermo else \'absdist\')(self.ptr(a),self.ptr(b),len(a),len(b),self.ptr(out));return out\n    def mnn(self,a,b,thermo=False):\n        if len(a)==0 or len(b)==0:return np.empty(0,int),np.empty(0,int)\n        row=np.empty(len(a),np.int32);col=np.empty(len(b),np.int32)\n        getattr(self.lib,\'thermomnn\' if thermo else \'absmnn\')(self.ptr(a),self.ptr(b),len(a),len(b),self.ptr(row),self.ptr(col))\n        ids=np.flatnonzero(col[row]==np.arange(len(a)));return ids,row[ids]\n\ndef self_test(engine):\n    rng=np.random.default_rng(SEED)\n    a=rng.integers(0,8,(35,64),dtype=np.uint8);b=rng.integers(0,8,(31,64),dtype=np.uint8)\n    # Force ties to verify deterministic first-index MNN rules.\n    a[1]=a[0];b[1]=b[0]\n    assert np.array_equal(a,engine.unpack3(engine.pack3(a)))\n    expected=np.abs(a[:,None,:].astype(np.int16)-b[None,:,:].astype(np.int16)).sum(-1)\n    assert np.array_equal(engine.distance(a,b),expected)\n    at,bt=engine.thermo(a),engine.thermo(b)\n    assert np.array_equal(engine.distance(at,bt,True),expected)\n    row=expected.argmin(1);col=expected.argmin(0);ids=np.flatnonzero(col[row]==np.arange(len(a)))\n    for aa,bb,t in [(a,b,False),(at,bt,True)]:\n        i,j=engine.mnn(aa,bb,t);assert np.array_equal(i,ids) and np.array_equal(j,row[ids])\n    # Exhaustively test all bin pairs and the sign bit extraction.\n    q=np.repeat(np.arange(8,dtype=np.uint8)[:,None],64,axis=1)\n    assert np.array_equal(engine.distance(engine.thermo(q),engine.thermo(q),True),64*np.abs(np.arange(8)[:,None]-np.arange(8)[None,:]))\n    x=rng.normal(size=(20,64)).astype(np.float32);pos=[31,25,24,23,22,21,20,19]\n    lut=rng.integers(0,8,(64,256),dtype=np.uint8)\n    assert np.array_equal(engine.encode(x,pos,lut),encode_np(x,pos,lut))\n    print(\'PASS: bit extraction, 3-bit roundtrip, all-bin ABS/popcount equality, full matrices, MNN including ties\',flush=True)\n\ndef project(k,H):\n    p=np.column_stack([k,np.ones(len(k))])@H.T\n    return (p[:,:2]/p[:,2:]).astype(np.float32)\n\ndef geometric_pairs(pair):\n    from scipy.spatial.distance import cdist\n    d=cdist(project(pair[\'k1\'],pair[\'H\']),pair[\'k2\']);back=cdist(project(pair[\'k2\'],np.linalg.inv(pair[\'H\'])),pair[\'k1\'])\n    j=d.argmin(1);i=np.arange(len(j));good=(d[i,j]<3)&(back[j,i]<3)&(back.argmin(1)[j]==i)\n    return i[good],j[good],d\n\ndef build_triplets(pairs):\n    result=[[],[],[]];cal=[];rng=np.random.default_rng(SEED)\n    for pair in pairs:\n        i,j,geo=geometric_pairs(pair)\n        if not len(i):continue\n        if len(i)>300:\n            sub=rng.choice(len(i),300,False);i,j=i[sub],j[sub]\n        # Hard negatives: most cosine-similar target at least 10 px from projected anchor.\n        sim=pair[\'d1\'][i]@pair[\'d2\'].T;sim[geo[i]<=10]=-np.inf\n        keep=np.isfinite(sim.max(1));i,j,sim=i[keep],j[keep],sim[keep]\n        if not len(i):continue\n        neg=sim.argmax(1)\n        for dest,x in zip(result,[pair[\'d1\'][i],pair[\'d2\'][j],pair[\'d2\'][neg]]):dest.append(x)\n        cal.extend([pair[\'d1\'],pair[\'d2\']])\n    if not result[0]:raise RuntimeError(\'No valid geometric triplets; inspect images and extraction.\')\n    return tuple(np.concatenate(x) for x in result),np.concatenate(cal)\n\ndef load_xfeat(root,top_k=1000):\n    import torch\n    root=Path(root);repo=root/\'xfeat_repo\'\n    if not repo.exists():\n        subprocess.run([\'git\',\'clone\',\'https://github.com/verlab/accelerated_features.git\',str(repo)],check=True)\n        subprocess.run([\'git\',\'-C\',str(repo),\'checkout\',XFEAT_COMMIT],check=True)\n    commit=subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip()\n    if commit!=XFEAT_COMMIT:raise RuntimeError(\'XFeat checkout differs from pinned experiment commit.\')\n    sys.path.insert(0,str(repo.resolve()))\n    from modules.xfeat import XFeat\n    torch.set_num_threads(2);torch.manual_seed(SEED)\n    model=XFeat(top_k=top_k)\n    for p in model.parameters():p.requires_grad_(False)\n    return model,commit\n\ndef extract(model,image):\n    import torch,cv2\n    t=torch.from_numpy(cv2.cvtColor(image,cv2.COLOR_BGR2RGB).copy()).permute(2,0,1)[None].float().to(model.dev)/255.\n    with torch.inference_mode():f=model.detectAndCompute(t,top_k=1000)[0]\n    d=f[\'descriptors\'].cpu().numpy().astype(np.float32)\n    d/=np.maximum(np.linalg.norm(d,axis=1,keepdims=True),1e-12)\n    return f[\'keypoints\'].cpu().numpy().astype(np.float32),np.ascontiguousarray(d)\n\ndef make_pairs(images,model,warps,seed):\n    import cv2\n    rng=np.random.default_rng(seed);pairs=[]\n    for name,img in images:\n        h,w=img.shape[:2];k1,d1=extract(model,img)\n        for r in range(warps):\n            aff=cv2.getRotationMatrix2D((w/2,h/2),float(rng.uniform(-20,20)),float(rng.uniform(.88,1.08)))\n            aff[:,2]+=rng.uniform(-.06,.06,2)*[w,h]\n            H=np.vstack([aff,[rng.uniform(-7e-5,7e-5),rng.uniform(-7e-5,7e-5),1]]).astype(np.float32)\n            im=cv2.warpPerspective(img,H,(w,h),borderMode=cv2.BORDER_CONSTANT)\n            im=np.clip(im.astype(np.float32)*rng.uniform(.82,1.12)+rng.uniform(-12,18),0,255).astype(np.uint8)\n            k2,d2=extract(model,im)\n            if min(len(d1),len(d2))<2:continue\n            pairs.append(dict(name=f\'{name}/warp{r}\',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n        print(\'extract\',name,len(d1),flush=True)\n    return pairs\n\ndef resize_decode(raw,max_side=640):\n    import cv2\n    im=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n    if im is None:raise ValueError(\'Unreadable image\')\n    h,w=im.shape[:2];s=min(1,max_side/max(h,w));return cv2.resize(im,(max(64,int(w*s)//32*32),max(64,int(h*s)//32*32)))\n\ndef read_coco(zip_path,count=24):\n    with zipfile.ZipFile(zip_path) as z:\n        names=sorted(n for n in z.namelist() if n.lower().endswith(\'.jpg\'))\n        if len(names)<count:raise ValueError(\'Not enough COCO images\')\n        chosen=np.random.default_rng(SEED).choice(len(names),count,False)\n        return [(Path(names[i]).name,resize_decode(z.read(names[i]))) for i in chosen]\n\ndef read_local_images(folder,count=24):\n    files=sorted(p for p in Path(folder).glob(\'*\') if p.suffix.lower() in [\'.jpg\',\'.jpeg\',\'.png\'])\n    if len(files)<count:raise ValueError(f\'Need {count} distinct images; found {len(files)}\')\n    return [(p.name,resize_decode(p.read_bytes())) for p in files[:count]]\n\ndef save_pair_cache(path,pairs):\n    payload={}\n    for i,p in enumerate(pairs):\n        for k,v in p.items():payload[f\'{i}_{k}\']=np.array(v)\n    payload[\'count\']=np.array(len(pairs));np.savez_compressed(path,**payload)\n\ndef read_pair_cache(path):\n    with np.load(path,allow_pickle=False) as z:\n        return [{k:(str(z[f\'{i}_{k}\']) if k==\'name\' else z[f\'{i}_{k}\']) for k in [\'name\',\'d1\',\'d2\',\'k1\',\'k2\',\'H\']} for i in range(int(z[\'count\']))]\n\ndef benchmark(fn,repeats=9):\n    for _ in range(3):fn()\n    vals=[]\n    for _ in range(repeats):\n        s=time.perf_counter();fn();vals.append((time.perf_counter()-s)*1000)\n    return float(np.median(vals))\n\ndef evaluate(pairs,model,engine,pos,lut,edges,out,label=\'heldout\'):\n    import torch\n    records=[]\n    for p in pairs:\n        a,b=[engine.encode(p[k],pos,lut) for k in [\'d1\',\'d2\']]\n        aa,bb=engine.pack3(a),engine.pack3(b);at,bt=engine.thermo(a),engine.thermo(b)\n        mi,mj=engine.mnn(a,b);ti,tj=engine.mnn(at,bt,True)\n        assert np.array_equal(mi,ti) and np.array_equal(mj,tj)\n        assert np.array_equal(engine.distance(a[:32],b[:32]),engine.distance(at[:32],bt[:32],True))\n        fa,fb=torch.from_numpy(p[\'d1\']),torch.from_numpy(p[\'d2\'])\n        def official():\n            with torch.inference_mode():return model.match(fa,fb,min_cossim=-1)\n        fi,fj=official();fi,fj=fi.numpy(),fj.numpy()\n        qa,qb=[bins_from_edges(p[k],edges) for k in [\'d1\',\'d2\']];qi,qj=engine.mnn(qa,qb)\n        for method,i,j in [(\'official_float_cosine\',fi,fj),(\'fixed_quantile_abs\',qi,qj),(\'learned_bits_abs\',mi,mj),(\'learned_bits_thermometer\',ti,tj)]:\n            err=np.linalg.norm(project(p[\'k1\'][i],p[\'H\'])-p[\'k2\'][j],axis=1)\n            gt1,gt2,_=geometric_pairs(p);gt=set(zip(gt1.tolist(),gt2.tolist()));hits=len(gt&set(zip(i.tolist(),j.tolist())))\n            records.append(dict(dataset=label,pair=p[\'name\'],method=method,n=len(a),m=len(b),mutual=len(i),correct=int((err<3).sum()),\n                precision=float((err<3).mean()*100) if len(err) else 0.,median_px=float(np.median(err)) if len(err) else None,\n                geometric_pair_recall=float(hits/len(gt)) if gt else None))\n        def encode_abs():\n            a,b=[engine.encode(p[k],pos,lut) for k in [\'d1\',\'d2\']];return engine.mnn(a,b)\n        def encode_packed3():\n            a,b=[engine.pack3(engine.encode(p[k],pos,lut)) for k in [\'d1\',\'d2\']]\n            return engine.mnn(engine.unpack3(a),engine.unpack3(b))\n        def encode_thermo():\n            a,b=[engine.thermo(engine.encode(p[k],pos,lut)) for k in [\'d1\',\'d2\']];return engine.mnn(a,b,True)\n        timings=dict(pair=p[\'name\'],n=len(a),m=len(b),official_float_mnn_ms=benchmark(official),\n            bin_encode_both_ms=benchmark(lambda:(engine.encode(p[\'d1\'],pos,lut),engine.encode(p[\'d2\'],pos,lut))),\n            abs_preencoded_mnn_ms=benchmark(lambda:engine.mnn(a,b)),\n            abs_packed3_preencoded_mnn_including_unpack_ms=benchmark(lambda:engine.mnn(engine.unpack3(aa),engine.unpack3(bb))),\n            thermometer_preencoded_mnn_ms=benchmark(lambda:engine.mnn(at,bt,True)),\n            abs_encode_mnn_ms=benchmark(encode_abs),abs_encode_pack_unpack_mnn_ms=benchmark(encode_packed3),\n            thermometer_encode_pack_mnn_ms=benchmark(encode_thermo))\n        for r in records[-4:]:r[\'timings\']=timings\n        print(\'eval\',label,p[\'name\'],[(r[\'method\'],round(r[\'precision\'],2),r[\'mutual\']) for r in records[-4:]],flush=True)\n    (out/f\'{label}_results.json\').write_text(json.dumps(records,indent=2));return records\n\ndef hpatches_pairs(zip_path,model,max_sequences=8):\n    import cv2\n    pairs=[]\n    with zipfile.ZipFile(zip_path) as z:\n        names=set(z.namelist());refs=sorted(n for n in names if n.endswith(\'/1.ppm\'))[:max_sequences]\n        for ref in refs:\n            directory=ref.rsplit(\'/\',1)[0];im1=cv2.imdecode(np.frombuffer(z.read(ref),np.uint8),cv2.IMREAD_COLOR)\n            k1,d1=extract(model,im1)\n            for index in range(2,7):\n                imname=f\'{directory}/{index}.ppm\';hname=f\'{directory}/H_1_{index}\'\n                if imname not in names or hname not in names:continue\n                im2=cv2.imdecode(np.frombuffer(z.read(imname),np.uint8),cv2.IMREAD_COLOR);k2,d2=extract(model,im2)\n                H=np.loadtxt(z.open(hname)).astype(np.float32)\n                if min(len(d1),len(d2))>=2:pairs.append(dict(name=f\'{directory}/1-{index}\',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n    if not pairs:raise ValueError(\'HPatches archive missing expected sequence images/homographies\')\n    return pairs\n\ndef run(root,coco_zip=None,image_dir=None,count=24,warps=3,search_samples=2000,sweeps=2,hpatches=None):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True);out=root/\'run\';out.mkdir(exist_ok=True)\n    engine=Engine(root);self_test(engine);model,commit=load_xfeat(root)\n    images=sorted(read_coco(coco_zip,count) if coco_zip else read_local_images(image_dir,count),key=lambda v:v[0])\n    order=np.random.default_rng(SEED).permutation(count);nt=max(2,int(count*.67));nv=max(1,int(count*.16))\n    groups=dict(train=order[:nt],validation=order[nt:nt+nv],test=order[nt+nv:])\n    manifest=dict(seed=SEED,source=\'COCO val2017\',groups={k:[images[i][0] for i in ids] for k,ids in groups.items()},\n                  xfeat_commit=commit,top_k=1000,max_side=640,warps_per_image=warps,\n                  note=\'image-disjoint synthetic homographies; COCO val2017 is our training source, not a COCO official benchmark\')\n    manifest[\'cache_fingerprint\']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()+b\'\'.join(hashlib.sha256(im.tobytes()).digest() for _,im in images)).hexdigest()\n    manifest_path=out/\'data_manifest.json\'\n    if manifest_path.exists() and json.loads(manifest_path.read_text()).get(\'cache_fingerprint\')!=manifest[\'cache_fingerprint\']:\n        raise RuntimeError(\'Existing pair cache belongs to a different dataset/config. Use a new experiment root or remove only its run/*_pairs.npz and data_manifest.json.\')\n    manifest_path.write_text(json.dumps(manifest,indent=2))\n    pairs={}\n    extraction=time.perf_counter()\n    for offset,(name,ids) in enumerate(groups.items()):\n        cache=out/f\'{name}_pairs.npz\'\n        if cache.exists():pairs[name]=read_pair_cache(cache)\n        else:\n            pairs[name]=make_pairs([images[i] for i in ids],model,warps,SEED+offset);save_pair_cache(cache,pairs[name])\n    extraction=time.perf_counter()-extraction\n    triplets,cal=build_triplets(pairs[\'train\'])\n    np.savez_compressed(out/\'training_triplets.npz\',anchor=triplets[0],positive=triplets[1],negative=triplets[2])\n    pos,lut,edges,training=train_codec(triplets,cal,out,search_samples=search_samples,sweeps=sweeps)\n    assert np.array_equal(engine.encode(cal[:100],pos,lut),encode_np(cal[:100],pos,lut))\n    records=[]\n    for name in [\'validation\',\'test\']:records+=evaluate(pairs[name],model,engine,pos,lut,edges,out,name)\n    hp_status=\'not run\'\n    if hpatches:\n        records+=evaluate(hpatches_pairs(hpatches,model),model,engine,pos,lut,edges,out,\'hpatches\');hp_status=\'run, never used for training\'\n    report=dict(training=training,manifest=manifest,extraction_seconds=extraction,hpatches=hp_status,\n        environment=dict(platform=platform.platform(),python=sys.version,numpy=np.__version__,cpp_flags=engine.flags,cpp_threads=1,torch_threads=2),\n        storage_bytes=dict(float32=256,selected_bit_intermediate=64,bin_uint8_working=64,bin_packed3=24,thermometer_packed=56,\n                           shared_lut=lut.nbytes,selected_positions=np.array(pos,np.int32).nbytes),\n        accuracy_equivalence=\'ABS and thermometer must match exactly, including deterministic tie-breaking\',\n        timing_scope=\'CPU encode/pack/matching; XFeat extraction excluded and reported separately; no claimed SIMD4 optimization\',\n        records=records)\n    (out/\'results.json\').write_text(json.dumps(report,indent=2))\n    import csv\n    with (out/\'accuracy.csv\').open(\'w\',newline=\'\') as f:\n        rows=[{k:v for k,v in r.items() if k!=\'timings\'} for r in records];w=csv.DictWriter(f,rows[0].keys());w.writeheader();w.writerows(rows)\n    with (out/\'timing.csv\').open(\'w\',newline=\'\') as f:\n        rows=[dict(dataset=r[\'dataset\'],**r[\'timings\']) for r in records if r[\'method\']==\'learned_bits_abs\']\n        w=csv.DictWriter(f,rows[0].keys());w.writeheader();w.writerows(rows)\n    print(\'Saved:\',out,flush=True);return report\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--root\',default=str(Path(__file__).parent));ap.add_argument(\'--coco-zip\');ap.add_argument(\'--image-dir\')\n    ap.add_argument(\'--count\',type=int,default=24);ap.add_argument(\'--warps\',type=int,default=3);ap.add_argument(\'--search-samples\',type=int,default=2000)\n    ap.add_argument(\'--sweeps\',type=int,default=2);ap.add_argument(\'--hpatches\');ap.add_argument(\'--self-test\',action=\'store_true\');args=ap.parse_args()\n    if args.self_test:self_test(Engine(args.root))\n    else:run(args.root,args.coco_zip,args.image_dir,args.count,args.warps,args.search_samples,args.sweeps,args.hpatches)\n', 'bit_codec.cpp': '#include <cstdint>\n#include <cstring>\n#include <vector>\n#include <cstdlib>\n#include <algorithm>\nextern "C" {\nvoid encode(const float* x,int n,const int* pos,int k,const uint8_t* lut,uint8_t* out){\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  uint32_t u; std::memcpy(&u,x+i*64+d,4); unsigned c=0;\n  for(int r=0;r<k;r++) c|=((u>>pos[r])&1u)<<r;\n  out[i*64+d]=lut[d*(1<<k)+c];\n }\n}\nvoid pack3(const uint8_t* x,int n,uint8_t* out){\n std::memset(out,0,n*24);\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  unsigned b=d*3, v=x[i*64+d]; out[i*24+b/8]|=v<<(b%8);\n  if(b%8>5) out[i*24+b/8+1]|=v>>(8-b%8);\n }\n}\nvoid unpack3(const uint8_t* x,int n,uint8_t* out){\n for(int i=0;i<n;i++) for(int d=0;d<64;d++){\n  unsigned b=d*3,v=x[i*24+b/8]>>(b%8);\n  if(b%8>5) v|=unsigned(x[i*24+b/8+1])<<(8-b%8);\n  out[i*64+d]=v&7;\n }\n}\nvoid packthermo(const uint8_t* x,int n,uint64_t* out){\n for(int i=0;i<n;i++) for(int r=0;r<7;r++){\n  uint64_t mask=0;for(int d=0;d<64;d++) if(x[i*64+d]>r) mask|=uint64_t(1)<<d;\n  out[i*7+r]=mask;\n }\n}\nvoid absdist(const uint8_t* a,const uint8_t* b,int n,int m,uint16_t* out){\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int d=0;d<64;d++)s+=std::abs(int(a[i*64+d])-int(b[j*64+d]));out[i*m+j]=s;\n }\n}\nvoid thermodist(const uint64_t* a,const uint64_t* b,int n,int m,uint16_t* out){\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int r=0;r<7;r++)s+=__builtin_popcountll(a[i*7+r]^b[j*7+r]);out[i*m+j]=s;\n }\n}\nvoid absmnn(const uint8_t* a,const uint8_t* b,int n,int m,int* row,int* col){\n std::vector<unsigned> bestrow(n,10000),bestcol(m,10000);\n std::fill(row,row+n,-1);std::fill(col,col+m,-1);\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int d=0;d<64;d++)s+=std::abs(int(a[i*64+d])-int(b[j*64+d]));\n  if(s<bestrow[i]){bestrow[i]=s;row[i]=j;}if(s<bestcol[j]){bestcol[j]=s;col[j]=i;}\n }\n}\nvoid thermomnn(const uint64_t* a,const uint64_t* b,int n,int m,int* row,int* col){\n std::vector<unsigned> bestrow(n,10000),bestcol(m,10000);\n std::fill(row,row+n,-1);std::fill(col,col+m,-1);\n for(int i=0;i<n;i++)for(int j=0;j<m;j++){\n  unsigned s=0;for(int r=0;r<7;r++)s+=__builtin_popcountll(a[i*7+r]^b[j*7+r]);\n  if(s<bestrow[i]){bestrow[i]=s;row[i]=j;}if(s<bestcol[j]){bestcol[j]=s;col[j]=i;}\n }\n}\n}\n', 'optimized.py': 'import numpy as np, ctypes, tempfile, subprocess\nfrom pathlib import Path\nCPU_THREADS=2\nAVX2=True\nptr=ctypes.c_void_p;integer=ctypes.c_int\ndef adr(x):return x.ctypes.data_as(ptr)\ncpp_source=\'#include <cstdint>\\n#include <vector>\\n#include <algorithm>\\n#include <immintrin.h>\\n#include <omp.h>\\nextern "C" int has_avx(){return __builtin_cpu_supports("avx512f") && __builtin_cpu_supports("avx512vpopcntdq");}\\n__attribute__((target("popcnt"))) static inline int scalar(const uint64_t*a,const uint64_t*b){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[k]^b[k]);return d;}\\n__attribute__((target("avx512f,avx512vpopcntdq"))) static int wide(const uint64_t*a,const uint64_t*b){\\n __m512i x=_mm512_maskz_loadu_epi64(0x7f,a),y=_mm512_maskz_loadu_epi64(0x7f,b);\\n return (int)_mm512_reduce_add_epi64(_mm512_popcnt_epi64(_mm512_xor_si512(x,y)));\\n}\\nextern "C" void encode_pack(const float*x,const float*e,int n,uint8_t*c,uint64_t*p){\\n for(int i=0;i<n;i++){for(int k=0;k<7;k++)p[i*7+k]=0;\\n for(int d=0;d<64;d++){int v=0;for(int k=0;k<7;k++)v+=x[i*64+d]>e[d*7+k];c[i*64+d]=v;\\n for(int k=0;k<v;k++)p[i*7+k]|=uint64_t(1)<<d;}}\\n}\\nextern "C" void matrix(const uint64_t*a,const uint64_t*b,int n,int m,int threads,uint16_t*out){\\n #pragma omp parallel for num_threads(threads) schedule(static)\\n for(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=scalar(a+i*7,b+j*7);\\n}\\nextern "C" void fused(const uint64_t*a,const uint64_t*b,int n,int m,int mode,int threads,int*ab,int*ba){\\n if(n==0||m==0)return;\\n const bool avx=mode==1&&has_avx();\\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\\n #pragma omp parallel num_threads(threads)\\n {\\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\\n #pragma omp for schedule(static)\\n for(int i=0;i<n;i++){int best=449,bj=m;\\n for(int j=0;j<m;j++){int dist=0;\\n if(mode==2){for(int k=0;k<7;k++){dist+=__builtin_popcountll(a[i*7+k]^b[j*7+k]);\\n // Strict inequality preserves the smallest-index tie rule in both directions.\\n if(dist>best&&dist>ds[j])break;}}\\n else dist=avx?wide(a+i*7,b+j*7):scalar(a+i*7,b+j*7);\\n if(dist<best){best=dist;bj=j;}\\n if(dist<ds[j]||(dist==ds[j]&&i<ids[j])){ds[j]=dist;ids[j]=i;}\\n }ab[i]=bj;}\\n }\\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){\\n int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\\n}\\nextern "C" int has_avx2(){return __builtin_cpu_supports("avx2");}\\nextern "C" void direct_scalar(const float*x,const float*et,int n,uint64_t*p){\\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;for(int d=0;d<64;d++)w|=uint64_t(x[i*64+d]>et[k*64+d])<<d;p[i*7+k]=w;}\\n}\\nextern "C" __attribute__((target("avx2"))) void direct_avx(const float*x,const float*et,int n,uint64_t*p){\\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;\\n for(int d=0;d<64;d+=8){auto a=_mm256_loadu_ps(x+i*64+d),b=_mm256_loadu_ps(et+k*64+d);\\n w|=uint64_t(uint32_t(_mm256_movemask_ps(_mm256_cmp_ps(a,b,_CMP_GT_OQ))))<<d;}p[i*7+k]=w;}\\n}\\n__attribute__((target("avx2"))) static inline __m256i counts4(__m256i v){\\n const __m256i lut=_mm256_setr_epi8(0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4,0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4);\\n auto mask=_mm256_set1_epi8(15);\\n auto lo=_mm256_shuffle_epi8(lut,_mm256_and_si256(v,mask));\\n auto hi=_mm256_shuffle_epi8(lut,_mm256_and_si256(_mm256_srli_epi16(v,4),mask));\\n return _mm256_sad_epu8(_mm256_add_epi8(lo,hi),_mm256_setzero_si256());\\n}\\nextern "C" void transpose7(const uint64_t*p,int n,uint64_t*s){for(int k=0;k<7;k++)for(int i=0;i<n;i++)s[k*n+i]=p[i*7+k];}\\nextern "C" __attribute__((target("avx2"))) void fused4(const uint64_t*a,const uint64_t*bs,int n,int m,int threads,int*ab,int*ba){\\n if(n==0||m==0)return;\\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\\n #pragma omp parallel num_threads(threads)\\n {\\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\\n #pragma omp for schedule(static)\\n for(int i=0;i<n;i++){int best=449,bj=m;int j=0;\\n for(;j+4<=m;j+=4){auto sum=_mm256_setzero_si256();\\n for(int k=0;k<7;k++){auto av=_mm256_set1_epi64x(a[i*7+k]);auto bv=_mm256_loadu_si256((const __m256i*)(bs+k*m+j));sum=_mm256_add_epi64(sum,counts4(_mm256_xor_si256(av,bv)));}\\n uint64_t dist[4];_mm256_storeu_si256((__m256i*)dist,sum);\\n for(int l=0;l<4;l++){int d=dist[l],q=j+l;if(d<best){best=d;bj=q;}if(d<ds[q]||(d==ds[q]&&i<ids[q])){ds[q]=d;ids[q]=i;}}\\n }\\n for(;j<m;j++){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[i*7+k]^bs[k*m+j]);if(d<best){best=d;bj=j;}if(d<ds[j]||(d==ds[j]&&i<ids[j])){ds[j]=d;ids[j]=i;}}\\n ab[i]=bj;}\\n }\\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\\n}\\n\'\nclass ThermometerVariant:\n def __init__(self, bins, edges):\n  self.bins=bins;self.planes=bins-1;self.bytes_per_kp=self.planes*8\n  self.name=f\'{bins} bins / {self.planes} bits: method 1+2+3 SIMD4\' if AVX2 else f\'{bins} bins / {self.planes} bits: scalar fallback\'\n  self.edges=np.ascontiguousarray(edges,dtype=np.float32)\n  self.et=np.ascontiguousarray(self.edges.T)\n  import re\n  source=re.sub(r\'\\b7\\b\',str(self.planes),cpp_source).replace(\'449\',str(64*self.planes+1))\n  # Original AVX512 distance loads at most eight planes: use exact scalar here.\n  # The benchmarked SIMD4 function is AVX2 and still processes ALL planes.\n  begin=source.index(\' __m512i x=\');end=source.index(\'\\n}\',begin)\n  source=source[:begin]+\' return scalar(a,b);\'+source[end:]\n  folder=Path(tempfile.mkdtemp(prefix=f\'thermometer_{bins}bin_\'))\n  (folder/\'kernel.cpp\').write_text(source)\n  subprocess.run([\'g++\',\'-O3\',\'-std=c++17\',\'-shared\',\'-fPIC\',\'-fopenmp\',\'-mpopcnt\',str(folder/\'kernel.cpp\'),\'-o\',str(folder/\'kernel.so\')],check=True)\n  self.lib=ctypes.CDLL(str(folder/\'kernel.so\'))\n  self.lib.has_avx2.restype=integer\n  if not self.lib.has_avx2():raise RuntimeError(\'This success method requires an AVX2 x86 CPU. Use a compatible CPU runtime.\')\n  for name in (\'direct_scalar\',\'direct_avx\'):getattr(self.lib,name).argtypes=[ptr,ptr,integer,ptr]\n  self.lib.transpose7.argtypes=[ptr,integer,ptr]\n  self.lib.fused4.argtypes=[ptr,ptr,integer,integer,integer,ptr,ptr]\n  self.lib.fused.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr,ptr]\n  self.lib.matrix.argtypes=[ptr,ptr,integer,integer,integer,ptr]\n  for name in (\'direct_scalar\',\'direct_avx\',\'transpose7\',\'fused4\',\'fused\',\'matrix\'):getattr(self.lib,name).restype=None\n def codes(self,x):return np.ascontiguousarray((x[:,:,None]>self.edges[None,:,:]).sum(axis=2),dtype=np.uint8)\n def encode(self,x,simd=True):\n  x=np.ascontiguousarray(x,dtype=np.float32);out=np.empty((len(x),self.planes),np.uint64)\n  fn=self.lib.direct_avx if simd and AVX2 else self.lib.direct_scalar\n  fn(adr(x),adr(self.et),len(x),adr(out));return out\n def soa(self,p):\n  out=np.empty((self.planes,len(p)),np.uint64);self.lib.transpose7(adr(p),len(p),adr(out));return out\n def match(self,a,bs):\n  nn,mm=len(a),bs.shape[1]\n  if not nn or not mm:return np.empty(0,np.int64),np.empty(0,np.int64)\n  ab=np.empty(nn,np.int32);ba=np.empty(mm,np.int32)\n  if AVX2:self.lib.fused4(adr(a),adr(bs),nn,mm,CPU_THREADS,adr(ab),adr(ba))\n  else:\n   b=np.ascontiguousarray(bs.T);self.lib.fused(adr(a),adr(b),nn,mm,0,CPU_THREADS,adr(ab),adr(ba))\n  ii=np.arange(nn);keep=ba[ab]==ii;return ii[keep],ab[keep].astype(np.int64)\n def matrix(self,a,b):\n  out=np.empty((len(a),len(b)),np.uint16);self.lib.matrix(adr(a),adr(b),len(a),len(b),CPU_THREADS,adr(out));return out\n def full(self,p):return self.match(self.encode(p[\'x\']),self.soa(self.encode(p[\'y\'])))\n', 'compare.py': 'from pathlib import Path\nimport json,time,ctypes,subprocess,platform\nimport numpy as np\nimport pandas as pd\nimport torch\nimport experiment as e\nfrom optimized import ThermometerVariant\n\nABS_CPP=r\'\'\'\n#include <cstdint>\n#include <vector>\n#include <algorithm>\n#include <immintrin.h>\n#include <omp.h>\nextern "C" void bins(const float*x,const float*edges,int n,uint8_t*out){\nfor(int i=0;i<n;i++)for(int d=0;d<64;d++){int v=0;for(int k=0;k<7;k++)v+=x[i*64+d]>edges[d*7+k];out[i*64+d]=v;}}\nextern "C" void match_abs(const uint8_t*a,const uint8_t*b,int n,int m,int threads,int*ab,int*ba){\nstd::vector<int> cd(threads*m,449),ci(threads*m,n);\n#pragma omp parallel num_threads(threads)\n{\nint t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n#pragma omp for schedule(static)\nfor(int i=0;i<n;i++){int best=449,bj=0;\nfor(int j=0;j<m;j++){__m128i sum=_mm_setzero_si128();\nfor(int d=0;d<64;d+=16){auto x=_mm_loadu_si128((const __m128i*)(a+i*64+d)),y=_mm_loadu_si128((const __m128i*)(b+j*64+d));\nsum=_mm_add_epi64(sum,_mm_sad_epu8(_mm_sub_epi8(_mm_max_epu8(x,y),_mm_min_epu8(x,y)),_mm_setzero_si128()));}\nuint64_t s[2];_mm_storeu_si128((__m128i*)s,sum);int v=s[0]+s[1];\nif(v<best){best=v;bj=j;}if(v<ds[j]||(v==ds[j]&&i<ids[j])){ds[j]=v;ids[j]=i;}}\nab[i]=bj;}}\nfor(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}}\n\'\'\'\nclass Abs:\n def __init__(self,root):\n  root=Path(root);(root/\'abs.cpp\').write_text(ABS_CPP)\n  subprocess.run([\'g++\',\'-O3\',\'-std=c++17\',\'-shared\',\'-fPIC\',\'-fopenmp\',str(root/\'abs.cpp\'),\'-o\',str(root/\'abs.so\')],check=True)\n  self.lib=ctypes.CDLL(str((root/\'abs.so\').resolve()));P=ctypes.c_void_p;I=ctypes.c_int\n  self.lib.bins.argtypes=[P,P,I,P];self.lib.match_abs.argtypes=[P,P,I,I,I,P,P]\n def bins(self,x,edges):\n  o=np.empty(x.shape,np.uint8);self.lib.bins(e.Engine.ptr(x),e.Engine.ptr(edges),len(x),e.Engine.ptr(o));return o\n def match(self,a,b):\n  if not len(a) or not len(b):return np.empty(0,int),np.empty(0,int)\n  ab=np.empty(len(a),np.int32);ba=np.empty(len(b),np.int32)\n  self.lib.match_abs(e.Engine.ptr(a),e.Engine.ptr(b),len(a),len(b),2,e.Engine.ptr(ab),e.Engine.ptr(ba))\n  i=np.flatnonzero(ba[ab]==np.arange(len(a)));return i,ab[i]\n\ndef compare(root,cache_root,model,training_seconds=None,quantile_seconds=None,pairs=None,timing_pairs_per_type=None):\n root=Path(root);cache_root=Path(cache_root);root.mkdir(exist_ok=True,parents=True)\n c=np.load(cache_root/\'learned_codec.npz\');pos=c[\'positions\'];lut=c[\'lut\'];edges=np.ascontiguousarray(c[\'edges\'])\n import shutil\n if not (root/\'bit_codec.cpp\').exists():shutil.copyfile(Path(e.__file__).parent/\'bit_codec.cpp\',root/\'bit_codec.cpp\')\n engine=e.Engine(root);e.self_test(engine);abs_engine=Abs(root);thermo=ThermometerVariant(8,edges)\n pairs=e.read_pair_cache(cache_root/\'test_pairs.npz\') if pairs is None else pairs\n timed_names={p[\'name\'] for p in pairs}\n if timing_pairs_per_type is not None:\n  timed_names=set();rng=np.random.default_rng(e.SEED)\n  for prefix in [\'i_\',\'v_\']:\n   names=[p[\'name\'] for p in pairs if p[\'name\'].split(\'/\')[-2].startswith(prefix)]\n   timed_names.update(rng.choice(names,min(timing_pairs_per_type,len(names)),replace=False).tolist())\n if training_seconds is None:training_seconds=json.loads((cache_root/\'training.json\').read_text())[\'seconds\']\n rng=np.random.default_rng(1)\n for n,m in [(1,1),(19,23),(13,3),(4,9)]:\n  a=rng.integers(0,8,(n,64),dtype=np.uint8);b=rng.integers(0,8,(m,64),dtype=np.uint8)\n  for tied in [False,True]:\n   if tied:a[:]=0;b[:]=0\n   ref=engine.mnn(a,b)\n   for got in [abs_engine.match(a,b),thermo.match(engine.thermo(a),thermo.soa(engine.thermo(b)))]:\n    for x,y in zip(ref,got):assert np.array_equal(x,y)\n records=[]\n for p in pairs:\n  a,b=p[\'d1\'],p[\'d2\'];qa,qb=abs_engine.bins(a,edges),abs_engine.bins(b,edges)\n  assert np.array_equal(engine.thermo(qa),thermo.encode(a))\n  la,lb=engine.encode(a,pos,lut),engine.encode(b,pos,lut)\n  methods={\n   \'float_cosine\':(lambda x:x,lambda x,y:model.match(torch.from_numpy(x),torch.from_numpy(y),min_cossim=-1),256,256,0,0),\n   \'quantile_thermometer_AVX2\':(thermo.encode,lambda x,y:thermo.match(x,thermo.soa(y)),56,56,edges.nbytes,quantile_seconds),\n   \'quantile_integer_ABS_8bin\':(lambda x:abs_engine.bins(x,edges),abs_engine.match,24,64,edges.nbytes,quantile_seconds),\n   \'learned_bits_integer_ABS\':(lambda x:engine.encode(x,pos,lut),abs_engine.match,24,64,lut.nbytes+pos.nbytes,training_seconds),\n   \'learned_bits_thermometer_AVX2\':(lambda x:engine.thermo(engine.encode(x,pos,lut)),lambda x,y:thermo.match(x,thermo.soa(y)),56,56,lut.nbytes+pos.nbytes,training_seconds)}\n  matches={}\n  for name,(enc,match,stored,working,shared,train) in methods.items():\n   x,y=enc(a),enc(b)\n   def full():return match(enc(a),enc(b))\n   with torch.inference_mode():\n    i,j=match(x,y);i=np.asarray(i);j=np.asarray(j);matches[name]=(i,j)\n    timed=p[\'name\'] in timed_names\n    total=e.benchmark(full) if timed else np.nan\n    encoding=(0 if name==\'float_cosine\' else e.benchmark(lambda:(enc(a),enc(b)))) if timed else np.nan\n    mnn=e.benchmark(lambda:match(x,y)) if timed else np.nan\n   packed_total=None\n   if \'integer_ABS\' in name and timed:\n    def packed():return abs_engine.match(engine.unpack3(engine.pack3(enc(a))),engine.unpack3(engine.pack3(enc(b))))\n    packed_total=e.benchmark(packed)\n   error=np.linalg.norm(e.project(p[\'k1\'][i],p[\'H\'])-p[\'k2\'][j],axis=1)\n   records.append(dict(pair=p[\'name\'],category=(\'illumination\' if p[\'name\'].split(\'/\')[-2].startswith(\'i_\') else \'viewpoint\' if p[\'name\'].split(\'/\')[-2].startswith(\'v_\') else \'synthetic\'),timing_measured=timed,method=name,mutual=len(i),correct=int((error<3).sum()),encode_ms=encoding,matching_ms=mnn,total_ms=total,packed_storage_total_ms=packed_total,stored_bytes_per_kp=stored,working_bytes_per_kp=working,shared_model_bytes=shared,offline_fit_seconds=train))\n  for first,second in [(\'quantile_integer_ABS_8bin\',\'quantile_thermometer_AVX2\'),(\'learned_bits_integer_ABS\',\'learned_bits_thermometer_AVX2\')]:\n   for x,y in zip(matches[first],matches[second]):assert np.array_equal(x,y)\n  print(\'done\',p[\'name\'],flush=True)\n df=pd.DataFrame(records);df.to_csv(root/\'per_pair_results.csv\',index=False)\n rows=[]\n for method,g in df.groupby(\'method\',sort=False):\n  r=dict(method=method,precision_pct=100*g.correct.sum()/g.mutual.sum(),mutual=int(g.mutual.sum()),correct=int(g.correct.sum()))\n  for k in [\'encode_ms\',\'matching_ms\',\'total_ms\',\'packed_storage_total_ms\']:r[k]=g[k].median()\n  for k in [\'stored_bytes_per_kp\',\'working_bytes_per_kp\',\'shared_model_bytes\',\'offline_fit_seconds\']:r[k]=g[k].iloc[0]\n  rows.append(r)\n summary=pd.DataFrame(rows);float_ms=summary.iloc[0].total_ms;summary[\'speedup_vs_float\']=float_ms/summary.total_ms\n for n in [1,1000,10000]:summary[f\'total_for_{n}_pairs_seconds\']=summary.offline_fit_seconds+summary.total_ms*n/1000\n summary.to_csv(root/\'comparison_summary.csv\',index=False)\n by_type=df.groupby([\'category\',\'method\'],sort=False).agg(mutual=(\'mutual\',\'sum\'),correct=(\'correct\',\'sum\'),total_ms=(\'total_ms\',\'median\'),timed_pairs=(\'timing_measured\',\'sum\')).reset_index()\n by_type[\'precision_pct\']=100*by_type.correct/by_type.mutual\n by_type.to_csv(root/\'by_type_summary.csv\',index=False)\n (root/\'environment.json\').write_text(json.dumps(dict(date=\'2026-10-11\',cpu=platform.processor(),platform=platform.platform(),torch_threads=torch.get_num_threads(),cpp_threads=2,selected_bits=pos.tolist(),scope=\'CPU after XFeat descriptor; same cached test pairs; transpose included in matching; scalar ABS uses SSE2/OpenMP, thermometer AVX2 SIMD4\',precision=\'pooled correct/mutual, homography error <3px\'),indent=2))\n print(summary.to_string(index=False));return summary\n', 'hpatches_eval.py': "from pathlib import Path\nimport json,time,zipfile,io,urllib.request\nimport numpy as np\nimport cv2\nimport experiment as e\nfrom compare import compare\n\nURL='https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'\ndef hpatches_all(zip_path,model,cache_dir):\n cache_dir=Path(cache_dir);cache_dir.mkdir(parents=True,exist_ok=True)\n if (cache_dir/'hpatches_pairs.npz').exists():return e.read_pair_cache(cache_dir/'hpatches_pairs.npz')\n pairs=[];manifest=[];start=time.perf_counter()\n def decode(raw):\n  orig=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n  if orig is None:raise ValueError('HPatches image decode failed')\n  img=e.resize_decode(raw,640)\n  scale=np.diag([img.shape[1]/orig.shape[1],img.shape[0]/orig.shape[0],1.]).astype(np.float64)\n  return img,scale\n with zipfile.ZipFile(zip_path) as z:\n  names=set(z.namelist());folders=sorted(n.rsplit('/',1)[0] for n in names if n.endswith('/1.ppm'))\n  for folder in folders:\n   scene=folder.split('/')[-1]\n   if not scene.startswith(('i_','v_')):continue\n   im,S1=decode(z.read(folder+'/1.ppm'));k1,d1=e.extract(model,im)\n   for idx in range(2,7):\n    raw=folder+f'/{idx}.ppm';hname=folder+f'/H_1_{idx}'\n    if raw not in names or hname not in names:raise ValueError(f'Missing image/homography: {scene}/{idx}')\n    im2,S2=decode(z.read(raw));k2,d2=e.extract(model,im2)\n    H0=np.loadtxt(io.BytesIO(z.read(hname)));H=S2@H0@np.linalg.inv(S1)\n    pairs.append(dict(name=f'{scene}/1-{idx}',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n   manifest.append(scene);print('extract',len(manifest),scene,flush=True)\n e.save_pair_cache(cache_dir/'hpatches_pairs.npz',pairs)\n (cache_dir/'hpatches_manifest.json').write_text(json.dumps(dict(scenes=manifest,n_scenes=len(manifest),n_pairs=len(pairs),illumination_scenes=sum(x.startswith('i_') for x in manifest),viewpoint_scenes=sum(x.startswith('v_') for x in manifest),max_side=640,top_k=1000,resize_homography='S2 @ H @ inv(S1)',source=URL,extraction_seconds=time.perf_counter()-start,protocol='all archive sequences, 1 versus 2..6; no excluded sequences; precision <3px, not official MMA benchmark',no_hpatches_training=True),indent=2))\n return pairs\n\ndef evaluate_hpatches(root,cache_root,model,zip_path,training_seconds=None,quantile_seconds=None):\n root=Path(root);root.mkdir(exist_ok=True,parents=True)\n pairs=hpatches_all(zip_path,model,root/'cache')\n return compare(root,cache_root,model,training_seconds,quantile_seconds,pairs=pairs,timing_pairs_per_type=12)\n\nif __name__=='__main__':\n root=Path(__file__).parent.resolve()\n model,commit=e.load_xfeat(root.parent/'learned_bits_experiment')\n import pandas as pd\n previous=pd.read_csv(root/'comparison_summary.csv').set_index('method')\n evaluate_hpatches(root/'hpatches',root/'run',model,'/tmp/hpatches_eval/hpatches.zip',float(previous.loc['learned_bits_integer_ABS','offline_fit_seconds']),float(previous.loc['quantile_integer_ABS_8bin','offline_fit_seconds']))\n"}
for name,content in sources.items():(ROOT/name).write_text(content)
sys.path.insert(0,str(ROOT))
# Remove stale imports if rerunning in an existing runtime.
for name in ['experiment','optimized','compare']:sys.modules.pop(name,None)
import experiment as e
from compare import compare
import numpy as np, torch, time, json, shutil
model,commit=e.load_xfeat(ROOT)
assert torch.get_num_threads()==2


In [ ]:
DATA_ROOT=ROOT/'data';DATA_ROOT.mkdir(exist_ok=True)
prep_start=time.perf_counter()
if REUSE_PREVIOUS_DESCRIPTORS and all((SOURCE_CACHE/f'{name}_pairs.npz').exists() for name in ['train','test']):
    train_pairs=e.read_pair_cache(SOURCE_CACHE/'train_pairs.npz')
    test_pairs=e.read_pair_cache(SOURCE_CACHE/'test_pairs.npz')
    manifest=json.loads((SOURCE_CACHE/'data_manifest.json').read_text())
    assert manifest['xfeat_commit']==commit, '캐시와 XFeat commit이 다릅니다.'
    assert not (set(manifest['groups']['train'])&set(manifest['groups']['test']))
    print('이전 descriptor 캐시 재사용:',SOURCE_CACHE)
else:
    assert COCO_ZIP.exists(),f'COCO ZIP을 찾을 수 없습니다: {COCO_ZIP}'
    images=sorted(e.read_coco(COCO_ZIP,24),key=lambda v:v[0])
    order=np.random.default_rng(e.SEED).permutation(24)
    groups={'train':order[:16],'validation':order[16:19],'test':order[19:]}
    manifest={'seed':e.SEED,'xfeat_commit':commit,'groups':{k:[images[i][0] for i in ids] for k,ids in groups.items()}}
    train_pairs=e.make_pairs([images[i] for i in groups['train']],model,3,e.SEED)
    test_pairs=e.make_pairs([images[i] for i in groups['test']],model,3,e.SEED+2)
e.save_pair_cache(DATA_ROOT/'train_pairs.npz',train_pairs)
e.save_pair_cache(DATA_ROOT/'test_pairs.npz',test_pairs)
(DATA_ROOT/'data_manifest.json').write_text(json.dumps(manifest,indent=2))
prep_seconds=time.perf_counter()-prep_start
triplet_start=time.perf_counter()
triplets,cal=e.build_triplets(train_pairs)
triplet_seconds=time.perf_counter()-triplet_start
s=time.perf_counter();edges=np.quantile(cal,np.arange(1,8)/8,axis=0).T.astype(np.float32)
quantile_seconds=time.perf_counter()-s
s=time.perf_counter()
pos,lut,edges,training=e.train_codec(triplets,cal,DATA_ROOT)
full_fit_seconds=time.perf_counter()-s
(ROOT/'preparation_timing.json').write_text(json.dumps(dict(preparation_seconds=prep_seconds,triplet_seconds=triplet_seconds,quantile_fit_seconds=quantile_seconds,learned_full_fit_seconds=full_fit_seconds,source_cache_reused=REUSE_PREVIOUS_DESCRIPTORS and SOURCE_CACHE.exists()),indent=2))
print('선택 비트:',pos,'전체 codec 학습:',full_fit_seconds,'초')


In [ ]:
summary=compare(ROOT,DATA_ROOT,model,full_fit_seconds,quantile_seconds)
from IPython.display import display
pd=__import__('pandas')
pd.set_option('display.max_columns',None)
display(summary.round(4))
print('CSV:',ROOT/'comparison_summary.csv')
print('쌍별 상세:',ROOT/'per_pair_results.csv')
print('같은 bin을 사용하는 ABS ↔ thermometer 거리/MNN 검사가 모두 통과했습니다.')


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(18,5))
labels=['float','quantile thermo','quantile ABS','learned ABS','learned thermo']
for ax,col,title in zip(axes,['total_ms','stored_bytes_per_kp','precision_pct'],['Encode + matching (ms), CPU','Stored bytes per descriptor','Precision (<3px), %']):
    ax.bar(labels,summary[col]);ax.set_title(title);ax.tick_params(axis='x',rotation=45)
fig.tight_layout();fig.savefig(ROOT/'comparison.png',dpi=180);plt.show()


`total_ms`는 각 쌍 반복 실행 중앙값들의 중앙값입니다. precision은 모든 test 쌍의 correct/mutual 합산입니다.
`matching_ms`에는 thermometer B 배열 transpose가 포함됩니다. 인코딩과 matching을 따로 재므로 두 값의 합이 total과 정확히 같지는 않습니다.
학습비 + 매칭 총비용은 측정 중앙값으로 계산한 추정이며, 데이터 다운로드·특징 추출·triplet 구성비는 `preparation_timing.json`에서 따로 확인합니다.
성능이 가장 좋은 표현은 CPU와 데이터에 따라 달라집니다. 이전 수치를 이번 결과로 재사용하지 않습니다.


## HPatches 전체 외부 평가 — 학습 규칙 고정
모든 116개 장면, 기준 이미지 1 → 2/3/4/5/6, 총 580쌍. HPatches로 학습·튜닝하지 않습니다.
최대 변 640, top_k 1000. 원본 homography는 리사이즈된 좌표로 보정합니다.
precision은 기존과 동일하게 correct (<3px) / mutual입니다. 논문의 공식 MMA 점수는 아닙니다.
8개 어려운 장면을 제외하는 별도 프로토콜은 적용하지 않고 전체 장면을 사용합니다.
정확도는 580쌍 전부 계산합니다. 속도 반복 측정은 seed로 선택한 조명 12쌍 + 시점 12쌍에서 수행하며 측정 목록도 저장합니다.


In [ ]:
RUN_HPATCHES=True
if RUN_HPATCHES:
    from hpatches_eval import evaluate_hpatches, URL
    import urllib.request
    HPATCHES_ZIP=BASE/'CSE527_datasets'/'hpatches-sequences-release.zip'
    if not HPATCHES_ZIP.exists():
        HPATCHES_ZIP.parent.mkdir(exist_ok=True,parents=True)
        partial=HPATCHES_ZIP.with_suffix('.zip.partial')
        urllib.request.urlretrieve(URL,partial)
        partial.replace(HPATCHES_ZIP)
    HP_ROOT=ROOT/'hpatches'
    hp_summary=evaluate_hpatches(HP_ROOT,DATA_ROOT,model,HPATCHES_ZIP,full_fit_seconds,quantile_seconds)
    display(hp_summary.round(4))
    hp_by_type=pd.read_csv(HP_ROOT/'by_type_summary.csv')
    display(hp_by_type.round(4))
    print('HPatches 전체:',HP_ROOT/'comparison_summary.csv')
    print('조명/시점별:',HP_ROOT/'by_type_summary.csv')
    print('전체 쌍별:',HP_ROOT/'per_pair_results.csv')
